In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
)

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score


In [ ]:
train_data_path = 'DataFolder/train_masked.csv'
val_data_path   = 'DataFolder/val_masked.csv'
test_data_path  = 'DataFolder/test_masked.csv'
df_train = pd.read_csv(train_data_path)
df_val  = pd.read_csv(val_data_path)
df_test = pd.read_csv(test_data_path)
df_train.head()

In [ ]:


class PairRedditRulesDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.labels = df["rule_violation"].astype(int).tolist()
        self.comment = df["body"].astype(str).tolist()
        self.context = (
            "RULE: " + df["rule"].astype(str) + "\n"
            "SUBREDDIT: " + df["subreddit"].astype(str) + "\n"
            "POS: " + df["positive_example_1"].fillna("").astype(str) + " || " +
                     df["positive_example_2"].fillna("").astype(str) + "\n"
            "NEG: " + df["negative_example_1"].fillna("").astype(str) + " || " +
                     df["negative_example_2"].fillna("").astype(str)
        ).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.comment[i],
            self.context[i],
            truncation="longest_first", 
            max_length=self.max_len,
            padding=False,
            return_tensors=None
        )
        enc["labels"] = int(self.labels[i])
        return enc


In [ ]:

model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True,  model_max_length=512)

train_pairds = PairRedditRulesDataset(df_train, tokenizer, max_len=512)
val_pairds = PairRedditRulesDataset(df_val, tokenizer, max_len=512)
test_pairds = PairRedditRulesDataset(df_test, tokenizer, max_len=512)
collate = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

train_dataloader = DataLoader(train_pairds, batch_size=64, 
                                shuffle=True,
                                collate_fn=collate)

val_dataloader = DataLoader(val_pairds, batch_size=64,
                            shuffle=False, 
                            collate_fn=collate)

batch = next(iter(train_dataloader))

In [ ]:
print(df_train.iloc[7])
print(train_pairds[7])
vocab = tokenizer.get_vocab()
print({k: v for k, v in vocab.items() if k in ['[CLS]', '[SEP]', '[PAD]', '[UNK]', '[MASK]']})

unused_count = 0
for k, v in vocab.items():
    if k.startswith('[unused'):
        unused_count += 1

print(f"Number of unused tokens in the tokenizer vocabulary: {unused_count} out of {len(vocab)} total tokens")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    probs_pos = (np.exp(logits) / np.exp(logits).sum(-1, keepdims=True))[:, 1]
    try:
        auroc = roc_auc_score(labels, probs_pos)
    except Exception:
        auroc = float("nan")
    acc = (preds == labels).mean()
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auroc": auroc}


In [ ]:
import transformers, tokenizers, sys, torch
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("python:", sys.version)
print("torch:", torch.__version__)
print("transformers path:", transformers.__file__)


In [ ]:

for p in model.base_model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

args_head = TrainingArguments(
    output_dir="./rb-rule-clf",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=20,
    learning_rate=3e-5,
    warmup_steps=200,                 
    weight_decay=0.03,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args_head,
    train_dataset=train_pairds,
    eval_dataset=val_pairds,
    tokenizer=tokenizer,
    data_collator=collate,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
from torch.optim import AdamW

encoder_layers = model.base_model.encoder.layer
for layer in encoder_layers[-8:]:
    for p in layer.parameters():
        p.requires_grad = True

encoder_params = []
for layer in encoder_layers[-8:]:
    encoder_params += list(layer.parameters())
head_params = list(model.classifier.parameters())

optimizer = AdamW([
    {"params": encoder_params, "lr": 3e-5},
    {"params": head_params,    "lr": 7e-4},
])

args_ft = TrainingArguments(
    output_dir="./rb-rule-clf-ft",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,         
    learning_rate=3e-5,         
    warmup_steps=200,
    weight_decay=0.03,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args_ft,
    train_dataset=train_pairds,
    eval_dataset=val_pairds,
    tokenizer=tokenizer,
    data_collator=collate,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None)
 
)

trainer.train()


In [ ]:
val_metrics = trainer.evaluate()
print(val_metrics)


In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_pairds)
print(test_metrics)

